# EMIT + Landsat 8/9 pairs: final notebook (Member 1, satellite data & preprocessing)

This notebook builds **EMIT hyperspectral + Landsat 8/9 pairs** for Kerala-like (humid tropical) cells and hands the passing pairs to Member 2.
It replaces the working notebook `EMIT_LANDSAT.ipynb`, which stays as the **archive**: all tests, previews and their outputs are kept there.

**Nothing here deletes data.** Every cell skips work that is already done, so running it on the existing Drive folders does not change finished results
(see the safety table below).

## Drive layout
```
MEMBER1___PREPROCESSING/
  handover_pairs.csv, screening/ranked_pairs.csv        inputs (notebook 1: pair screening)
  EMIT-LANDSAT/
    config/     emit_bands.json (wavelengths, kept bands), emit_export_bands.json (export band order)
    batch/      emit_gee_check.csv, emit_cell_stats.csv (one row per scene: handover + extra),
                emit_extra_stats.csv, emit_export_state.csv
    processed/  emitpair_<scene>.tif          raw Earth Engine exports (EMIT + flags + Landsat on the EMIT 60 m grid)
    pairs/      <N>/ emit_reflectance.tif, landsat_<id>.tif, pair_info.json      master archive (PASS and FAIL)
                pair_index.csv (pair numbers, append-only), emit_final_qc_<version>.csv
    PASSED-PAIRS/  copies of the PASS pairs for Member 2 + handover_pass.csv + README.txt
```

## Run order
| cell | what it does | when to run |
|---|---|---|
| **0 STATUS** | prints where everything stands (read-only) | any time, first |
| A1 | handover scenes: Earth Engine check + per-scene facts | done; skipped automatically |
| A2 | band list (232 of 285) + export band order | done; only checks |
| A3 | more scenes from the screening (pre-filter) | only to add more pairs (edit the caps) |
| A4 | submit Earth Engine exports + task status | after A3; re-run to check progress |
| **B1** | QC of all exports -> pair folders (EMIT QC v2.3) | when all exports are on Drive |
| **B2** | copy PASS pairs to PASSED-PAIRS + README | after every B1 run |

Normal use now: **0 -> B1 -> B2 -> 0**.

## Safety: what each cell writes
| cell | writes | can it lose data? |
|---|---|---|
| 0 | nothing | no |
| A1 | `emit_gee_check.csv`, `emit_cell_stats.csv`, `emit_bands.json` | no: **skipped if `emit_cell_stats.csv` exists** (the old E-02 had no such guard) |
| A2 | the two config files | no: writes only if missing; otherwise compares and never overwrites |
| A3 | appends to `emit_extra_stats.csv` and `emit_cell_stats.csv` | no: append-only, backup `emit_cell_stats_v1.csv` |
| A4 | Earth Engine tasks, `emit_export_state.csv` | no: skips scenes already submitted or on Drive |
| B1 | moves new exports to `processed/`; rewrites the files in `pairs/<N>/` | no: rebuilt from the raw exports, same input gives the same result; `pair_info_v1.json` kept |
| B2 | `PASSED-PAIRS/` | no: pairs that no longer pass are **moved** to `PASSED-PAIRS/_to_delete/`, never deleted |

## 0 STATUS
Read-only overview: scenes, exports, pair folders, QC version, PASS counts and whether `PASSED-PAIRS` matches the PASS list.

In [ ]:
# 0. STATUS: where the EMIT pipeline stands (read-only; changes nothing). Run any time.
import os, glob, json
import pandas as pd

# ---- idempotent setup ----
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"; ROOT = f"{BASE}/EMIT-LANDSAT"
PROC, PAIRS, BATCH, CONF = f"{ROOT}/processed", f"{ROOT}/pairs", f"{ROOT}/batch", f"{ROOT}/config"
for d in ("raw", "processed", "pairs", "batch", "config"): os.makedirs(f"{ROOT}/{d}", exist_ok=True)
OUT = f"{ROOT}/PASSED-PAIRS"

print("INPUTS")
for p in (f"{BASE}/handover_pairs.csv", f"{BASE}/screening/ranked_pairs.csv"):
    print(f"  {'ok     ' if os.path.exists(p) else 'MISSING'} {p.replace(BASE + '/', '')}")

print("\nPART A: scenes and exports")
sp = f"{BATCH}/emit_cell_stats.csv"
st = pd.read_csv(sp).drop_duplicates("gee_index") if os.path.exists(sp) else None
if st is not None:
    src = st["source"].fillna("handover") if "source" in st.columns else pd.Series("handover", index=st.index)
    print(f"  scenes (emit_cell_stats.csv): {len(st)} | by source: {src.value_counts().to_dict()} | cells: {st['cell_id'].nunique()}")
else: print("  emit_cell_stats.csv not found -> run A1")
for f in ("emit_bands.json", "emit_export_bands.json"):
    print(f"  config/{f}: {'ok' if os.path.exists(f'{CONF}/{f}') else 'MISSING -> run A1/A2'}")
ep = f"{BATCH}/emit_export_state.csv"
state = set(pd.read_csv(ep)["name"]) if os.path.exists(ep) else set()
in_proc = {os.path.basename(f)[:-4] for f in glob.glob(f"{PROC}/emitpair_*.tif")}
in_top = {os.path.basename(f)[:-4] for f in glob.glob(f"{BASE}/emitpair_*.tif")}
print(f"  exports submitted: {len(state)} | on Drive: {len(in_proc | in_top)} (not yet moved to processed/: {len(in_top)})"
      f" | submitted but not on Drive yet: {len(state - in_proc - in_top)}")
if st is not None:
    print(f"  scenes never submitted: {len({f'emitpair_{g}' for g in st['gee_index']} - state - in_proc - in_top)}")

print("\nPART B: pairs and QC")
rows = []
for d in glob.glob(f"{PAIRS}/*"):
    if not (os.path.isdir(d) and os.path.basename(d).isdigit()): continue
    jp = f"{d}/pair_info.json"
    if not os.path.exists(jp): rows.append({"pair": int(os.path.basename(d)), "status": "no pair_info.json"}); continue
    i = json.load(open(jp)); q = i.get("final_qc") or {}
    rows.append({"pair": int(os.path.basename(d)), "status": i.get("status"), "qc": i.get("qc_version"), "cell": i.get("cell_id"),
                 "climate": i.get("climate_group"), "haze_mild": bool(q.get("haze_mild") or False)})
pa = pd.DataFrame(rows, columns=["pair", "status", "qc", "cell", "climate", "haze_mild"])
if len(pa):
    print(f"  pair folders: {len(pa)} | status: {pa['status'].value_counts().to_dict()} | QC versions: {pa['qc'].value_counts(dropna=False).to_dict()}")
    pp = pa[pa["status"] == "PASS"]
    print(f"  PASS: {len(pp)} pairs in {pp['cell'].nunique()} cells | climate: {pp['climate'].value_counts(dropna=False).to_dict()}"
          f" | haze_mild: {int(pp['haze_mild'].sum())}")
else: print("  no pair folders yet -> run B1")

print("\nHANDOVER (PASSED-PAIRS)")
copies = {int(os.path.basename(d)) for d in glob.glob(f"{OUT}/*") if os.path.isdir(d) and os.path.basename(d).isdigit()}
want = set(pa.loc[pa["status"] == "PASS", "pair"]) if len(pa) else set()
print(f"  copies: {len(copies)} | PASS pairs not copied: {len(want - copies)} | copies that are no longer PASS: {len(copies - want)}")
print("  -> in sync" if want == copies else "  -> NOT in sync: run B2 (pairs in B2's EXCLUDE list also show up here)")

# PART A: scene selection and Earth Engine export
All of Part A is **done** (400 scenes exported). The cells are kept so the pipeline is complete and can be extended; each one skips finished work.

## A1 Handover scenes: Earth Engine check + per-scene facts
For the EMIT rows of `handover_pairs.csv`: finds each scene in Earth Engine by acquisition timestamp, then computes per-scene facts on the
7.7 km cell (sun elevation, AOD, water vapour, flag fractions, valid area, Landsat scene cloud). Writes `emit_cell_stats.csv` and the wavelength list.
Result on 20 Oct: 36 of 40 found (3 newer than the collection, 1 unexplained). **Skipped if `emit_cell_stats.csv` exists.** (= old E-01 + E-02)

In [ ]:
# A1. Handover EMIT scenes: Earth Engine check + per-scene facts (= old E-01 + E-02).
#     Runs only once: SKIPPED if batch/emit_cell_stats.csv exists (re-running would overwrite the extra scenes added by A3).
import os, re, json
import numpy as np, pandas as pd, ee

# ---- idempotent setup ----
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"; ROOT = f"{BASE}/EMIT-LANDSAT"
PROC, PAIRS, BATCH, CONF = f"{ROOT}/processed", f"{ROOT}/pairs", f"{ROOT}/batch", f"{ROOT}/config"
for d in ("raw", "processed", "pairs", "batch", "config"): os.makedirs(f"{ROOT}/{d}", exist_ok=True)
FLAGS = ["cloud_flag", "cirrus_flag", "water_flag", "spacecraft_flag", "dilated_cloud_flag", "aggregate_flag"]
sp = f"{BATCH}/emit_cell_stats.csv"

if os.path.exists(sp):
    print(f"A1 already done: emit_cell_stats.csv has {len(pd.read_csv(sp))} rows -> skipped, nothing changed.")
else:
    import ee
    EE_PROJECT, CH, HALF = "hyperspectral-thermal-511303", "NASA/EMIT/L2A/RFL", 0.035
    try: ee.Initialize(project=EE_PROJECT)
    except Exception: ee.Authenticate(); ee.Initialize(project=EE_PROJECT)
    stamp = lambda s: (lambda m: m.group(1) if m else None)(re.search(r"(\d{8}T\d{6})", str(s)))
    col = ee.ImageCollection(CH)
    em = pd.read_csv(f"{BASE}/handover_pairs.csv"); em = em[em["sensor"] == "EMIT"].reset_index(drop=True)
    print("EMIT rows in the handover list:", len(em), "| unique scenes:", em["scene_id"].nunique(), "| cells:", em["cell_id"].nunique())

    # 1) is each scene in Earth Engine? (same acquisition timestamp, image covering the cell centre)
    chk = []
    for n, r in enumerate(em.itertuples(), 1):
        t = pd.Timestamp(r.acq_datetime)
        c = col.filterBounds(ee.Geometry.Point([float(r.lon), float(r.lat)])).filterDate(
            (t - pd.Timedelta(days=1)).strftime("%Y-%m-%d"), (t + pd.Timedelta(days=2)).strftime("%Y-%m-%d"))
        ids = c.aggregate_array("system:index").getInfo(); hit = [i for i in ids if stamp(i) == stamp(r.scene_id)]
        chk.append({"scene_id": r.scene_id, "images_near": len(ids), "exact_match": bool(hit),
                    "gee_index": hit[0] if hit else (ids[0] if ids else None)})
        if n % 10 == 0: print(f"  checked {n}/{len(em)}")
    chk = pd.DataFrame(chk); chk.to_csv(f"{BATCH}/emit_gee_check.csv", index=False)
    ok = em.merge(chk.drop_duplicates("scene_id")[["scene_id", "exact_match", "gee_index"]], on="scene_id")
    ok = ok[ok["exact_match"]].reset_index(drop=True)
    print(f"found in Earth Engine: {len(ok)} of {len(em)}")

    # 2) per-scene facts, computed in Earth Engine on the cell (no downloads)
    def landsat_cc(lid):
        coll = "LANDSAT/LC09/C02/T1_L2" if lid.startswith("LC09") else "LANDSAT/LC08/C02/T1_L2"
        try: return float(ee.Image(f"{coll}/{lid}").get("CLOUD_COVER").getInfo())
        except Exception: return np.nan
    rows, wl0 = [], None
    for n, r in enumerate(ok.itertuples(), 1):
        img = ee.Image(f"{CH}/{r.gee_index}")
        reg = ee.Geometry.Rectangle([r.lon - HALF, r.lat - HALF, r.lon + HALF, r.lat + HALF], geodesic=False)
        b50 = img.select("reflectance_50")
        stack = img.select(["AOD550", "H2O", "elev"] + FLAGS).addBands(b50.gt(0).unmask(0).rename("refl_pos"))
        out = ee.Dictionary({"st": stack.reduceRegion(ee.Reducer.mean(), reg, 60, maxPixels=1e9),
                             "sz": img.get("SOLAR_ZENITH"), "sa": img.get("SOLAR_AZIMUTH"),
                             "wl": img.get("reflectance_wavelengths")}).getInfo()
        wl = np.array(out["wl"], float)
        if wl0 is None: wl0 = wl
        s = out["st"]; lid = re.sub(r"^\d+_", "", str(r.ls_ids))
        rows.append({"gee_index": r.gee_index, "date": re.search(r"(\d{8})T", r.gee_index).group(1), "cell_id": r.cell_id,
                     "lon": r.lon, "lat": r.lat, "ls_id": lid, "ls_dt": r.ls_dt,
                     "sun_elev": 90 - float(out["sz"]) if out["sz"] is not None else np.nan,
                     "sun_az": float(out["sa"]) if out["sa"] is not None else np.nan,
                     "aod": s.get("AOD550"), "h2o": s.get("H2O"), "elev": s.get("elev"),
                     "cloud": s.get("cloud_flag"), "cirrus": s.get("cirrus_flag"), "water": s.get("water_flag"),
                     "spacecraft": s.get("spacecraft_flag"), "dilated": s.get("dilated_cloud_flag"),
                     "aggregate": s.get("aggregate_flag"), "valid": s.get("refl_pos"),
                     "wl_same": bool(len(wl) == len(wl0) and np.max(np.abs(wl - wl0)) < 1e-3),
                     "ls_cc": landsat_cc(lid), "source": "handover"})
        if n % 10 == 0: print(f"  facts {n}/{len(ok)}")
    res = pd.DataFrame(rows)
    for c in res.columns.difference(["gee_index", "date", "cell_id", "ls_id", "wl_same", "source"]): res[c] = pd.to_numeric(res[c], errors="coerce")
    res.to_csv(sp, index=False)
    cfgp = f"{CONF}/emit_bands.json"
    if not os.path.exists(cfgp):
        json.dump({"n_bands": int(len(wl0)), "wavelength_nm": wl0.tolist(), "source": "GEE NASA/EMIT/L2A/RFL property reflectance_wavelengths"},
                  open(cfgp, "w"))
    print(f"\nsaved: emit_cell_stats.csv ({len(res)} scenes) | same wavelengths in all scenes: {bool(res['wl_same'].all())} | bands: {len(wl0)}")
    print(res[["sun_elev", "aod", "cloud", "water", "aggregate", "valid", "ls_cc"]].describe().round(2).T.to_string())

## A2 Band list and export band order
Drops the two water-vapour gaps (bands 127-141 and 187-212, found in the mean spectra of 36 scenes; old E-03) plus a 3-band margin:
**232 of 285 bands kept**. Also fixes the export band order (reflectance, 6 flags, AOD/H2O/elevation, 8 Landsat layers).
Writes the config files only if they are missing; otherwise it checks them and never overwrites. (= old E-04 part 1)

In [ ]:
# A2. Band list (232 of 285) + export band order (= old E-04 part 1). Writes only if missing; otherwise compares, never overwrites.
import os, json
import numpy as np

# ---- idempotent setup ----
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"; ROOT = f"{BASE}/EMIT-LANDSAT"
PROC, PAIRS, BATCH, CONF = f"{ROOT}/processed", f"{ROOT}/pairs", f"{ROOT}/batch", f"{ROOT}/config"
for d in ("raw", "processed", "pairs", "batch", "config"): os.makedirs(f"{ROOT}/{d}", exist_ok=True)
GAPS, MARGIN = [(127, 141), (187, 212)], 3                      # design choice: water-vapour gaps + 3 band margin
FLAGS = ["cloud_flag", "cirrus_flag", "water_flag", "dilated_cloud_flag", "spacecraft_flag", "aggregate_flag"]   # export order (E-04)
EXTRA = ["AOD550", "H2O", "elev"]
LSBANDS = ["blue", "green", "red", "nir", "swir1", "swir2", "LST_K", "clear_frac"]

cfgp, expp = f"{CONF}/emit_bands.json", f"{CONF}/emit_export_bands.json"
cfg = json.load(open(cfgp)); wl = np.array(cfg["wavelength_nm"])
drop = sorted({k for a, b in GAPS for k in range(a - MARGIN, b + MARGIN + 1)})
keep = [k for k in range(len(wl)) if k not in drop]
if "keep_band_indices" not in cfg:
    cfg.update({"gaps": GAPS, "margin_bands": MARGIN, "dropped_band_indices": drop, "keep_band_indices": keep,
                "note": "gaps from the mean spectra of 36 scenes (E-03); band index = GEE reflectance_<index>"})
    json.dump(cfg, open(cfgp, "w")); print("emit_bands.json: band list written")
else:
    print("emit_bands.json: band list", "unchanged" if cfg["keep_band_indices"] == keep else "DIFFERS from the saved one -> NOT overwritten")

exp_new = {"emit_reflectance_band_indices": keep, "emit_flags": FLAGS, "emit_extra": EXTRA, "landsat": LSBANDS,
           "order": "reflectance (len keep) + flags + extra + landsat", "nodata": -9999}
if not os.path.exists(expp):
    json.dump(exp_new, open(expp, "w")); print("emit_export_bands.json: written")
else:
    exp = json.load(open(expp))
    same = all(exp.get(k) == exp_new[k] for k in ("emit_reflectance_band_indices", "emit_flags", "emit_extra", "landsat"))
    print("emit_export_bands.json:", "unchanged" if same else "DIFFERS from the saved one -> NOT overwritten")
print(f"\nbands: {len(wl)} -> keep {len(keep)}, dropped {len(drop)}: "
      + ", ".join(f"{a - MARGIN}-{b + MARGIN} ({wl[a - MARGIN]:.0f}-{wl[b + MARGIN]:.0f} nm)" for a, b in GAPS))

## A3 More EMIT scenes from the screening
Takes the EMIT (scene, cell) rows of `screening/ranked_pairs.csv` that are not used yet, keeps the chosen profiles and at most
`MAX_DATES_PER_CELL` scenes per cell (counting those already used), finds them in Earth Engine, computes the same per-scene facts as A1
and pre-filters (sun >= 25 deg, AOD <= 0.8, valid area >= 0.5, cloud flag <= 0.3). Scenes marked `process` are **appended** to `emit_cell_stats.csv`.
Run on 22 Oct: 497 unused humid scenes -> 417 under the cap -> 397 in Earth Engine -> 378 unique -> **365 process**.
Resumable; re-running adds nothing unless you raise the cap or add profiles. (= old E-10)

In [ ]:
# A3. MORE EMIT SCENES: unused EMIT scenes from the screening -> Earth Engine availability -> per-scene facts -> append. Resumable. (= old E-10)
import os, re, json, shutil
import numpy as np, pandas as pd, ee

# ---- idempotent setup ----
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"; ROOT = f"{BASE}/EMIT-LANDSAT"
PROC, PAIRS, BATCH, CONF = f"{ROOT}/processed", f"{ROOT}/pairs", f"{ROOT}/batch", f"{ROOT}/config"
for d in ("raw", "processed", "pairs", "batch", "config"): os.makedirs(f"{ROOT}/{d}", exist_ok=True)
EE_PROJECT, CH, HALF = "hyperspectral-thermal-511303", "NASA/EMIT/L2A/RFL", 0.035
try: ee.Initialize(project=EE_PROJECT)
except Exception: ee.Authenticate(); ee.Initialize(project=EE_PROJECT)
FLAGS = ["cloud_flag", "cirrus_flag", "water_flag", "spacecraft_flag", "dilated_cloud_flag", "aggregate_flag"]

# ---- design choices (edit) ----
INCLUDE_PROFILES = ["humid_coastal"]             # Kerala-like first; add "dense_city", "semi_arid_city" later if wanted
MAX_DATES_PER_CELL = 4                           # scenes per cell, counting those already used
MIN_SUN, MAX_AOD, MIN_VALID, MAX_CLOUD = 25.0, 0.8, 0.5, 0.3
pd.set_option("display.width", 220)

# ---- 1) candidates: EMIT (scene, cell) pairs from the screening, minus scenes already used ----
stamp = lambda s: (lambda m: m.group(1) if m else None)(re.search(r"(\d{8}T\d{6})", str(s)))
r = pd.read_csv(f"{BASE}/screening/ranked_pairs.csv"); r = r[r["sensor"] == "EMIT"].copy()
st0 = pd.read_csv(f"{BATCH}/emit_cell_stats.csv")
ho = pd.read_csv(f"{BASE}/handover_pairs.csv"); ho = ho[ho["sensor"] == "EMIT"]
used = {stamp(x) for x in list(ho["scene_id"]) + list(st0["gee_index"])}
r["stamp"] = r["scene_id"].map(stamp); r = r[~r["stamp"].isin(used)].copy()
print(f"(1) unused EMIT (scene, cell) rows: {len(r)} | unique scenes by profile:", r.groupby("profile")["scene_id"].nunique().to_dict())
r = r[r["profile"].isin(INCLUDE_PROFILES)].copy()
r["prio"] = r["profile"].map({"humid_coastal": 0, "semi_arid_city": 1, "dense_city": 2})
r["abs_dt"] = r["ls_dt"].astype(float).abs(); r["cell_id"] = r["cell_id"].astype(str)
r["cloud_sort"] = r["cloud_pct"].fillna(50)
best = r.sort_values(["prio", "fit", "cloud_sort", "abs_dt"], ascending=[True, False, True, True]).drop_duplicates("scene_id").copy()
have = st0["cell_id"].astype(str).value_counts()
best["dates_before"] = best["cell_id"].map(have).fillna(0).astype(int)
best["rank_in_cell"] = best.groupby("cell_id").cumcount() + best["dates_before"]
print(f"    {INCLUDE_PROFILES}: unused scenes (one cell each) {len(best)} in {best['cell_id'].nunique()} cells "
      f"({int((best['dates_before'] == 0).groupby(best['cell_id']).first().sum())} cells never used)")
print("    scenes kept for each cap (max scenes per cell incl. already used):",
      {c: int((best["rank_in_cell"] < c).sum()) for c in (1, 2, 3, 4, 6, 10)})
todo = best[best["rank_in_cell"] < MAX_DATES_PER_CELL].copy()
print(f"    with MAX_DATES_PER_CELL = {MAX_DATES_PER_CELL}: {len(todo)} scenes in {todo['cell_id'].nunique()} cells")

# ---- 2) which of them does Earth Engine have? (exact id first, then by acquisition timestamp) ----
col = ee.ImageCollection(CH); ids = todo["scene_id"].tolist(); found = set()
for i in range(0, len(ids), 200):
    found |= set(col.filter(ee.Filter.inList("system:index", ids[i:i + 200])).aggregate_array("system:index").getInfo())
todo["gee_index"] = [s if s in found else None for s in todo["scene_id"]]
miss = todo[todo["gee_index"].isna()]
print(f"\n(2) exact id match in Earth Engine: {len(found)} | searching {len(miss)} by timestamp ...")
for i, rw in miss.iterrows():
    t = pd.Timestamp(rw["acq_datetime"])
    c = col.filterBounds(ee.Geometry.Point([float(rw["lon"]), float(rw["lat"])])).filterDate(
        (t - pd.Timedelta(days=1)).strftime("%Y-%m-%d"), (t + pd.Timedelta(days=2)).strftime("%Y-%m-%d"))
    hit = [x for x in c.aggregate_array("system:index").getInfo() if stamp(x) == rw["stamp"]]
    if hit: todo.at[i, "gee_index"] = hit[0]
print(f"    found in Earth Engine: {int(todo['gee_index'].notna().sum())} of {len(todo)} (missing: {int(todo['gee_index'].isna().sum())}, "
      f"e.g. newer than the collection)")
avail = todo[todo["gee_index"].notna()].drop_duplicates("gee_index")

# ---- 3) per-scene facts (as A1), resumable ----
wl_ref = np.array(json.load(open(f"{CONF}/emit_bands.json"))["wavelength_nm"])
fp = f"{BATCH}/emit_extra_stats.csv"
done = set(pd.read_csv(fp)["gee_index"]) if os.path.exists(fp) else set()
def landsat_coll(lid): return "LANDSAT/LC09/C02/T1_L2" if lid.startswith("LC09") else "LANDSAT/LC08/C02/T1_L2"
buf, n_new = [], 0
for n, rw in enumerate(avail.itertuples(), 1):
    if rw.gee_index in done: continue
    try:
        img = ee.Image(f"{CH}/{rw.gee_index}")
        reg = ee.Geometry.Rectangle([rw.lon - HALF, rw.lat - HALF, rw.lon + HALF, rw.lat + HALF], geodesic=False)
        b50 = img.select("reflectance_50")
        stack = img.select(["AOD550", "H2O", "elev"] + FLAGS).addBands(b50.gt(0).unmask(0).rename("refl_pos"))
        out = ee.Dictionary({"st": stack.reduceRegion(ee.Reducer.mean(), reg, 60, maxPixels=1e9),
                             "sz": img.get("SOLAR_ZENITH"), "sa": img.get("SOLAR_AZIMUTH"),
                             "wl": img.get("reflectance_wavelengths")}).getInfo()
        lid = re.sub(r"^\d+_", "", str(rw.ls_ids))
        try: cc = float(ee.Image(f"{landsat_coll(lid)}/{lid}").get("CLOUD_COVER").getInfo())
        except Exception: cc = np.nan
        s, wl = out["st"], np.array(out["wl"], float)
        buf.append({"gee_index": rw.gee_index, "date": re.search(r"(\d{8})T", rw.gee_index).group(1), "cell_id": rw.cell_id,
                    "lon": rw.lon, "lat": rw.lat, "ls_id": lid, "ls_dt": rw.ls_dt,
                    "sun_elev": 90 - float(out["sz"]) if out["sz"] is not None else np.nan,
                    "sun_az": float(out["sa"]) if out["sa"] is not None else np.nan,
                    "aod": s.get("AOD550"), "h2o": s.get("H2O"), "elev": s.get("elev"),
                    "cloud": s.get("cloud_flag"), "cirrus": s.get("cirrus_flag"), "water": s.get("water_flag"),
                    "spacecraft": s.get("spacecraft_flag"), "dilated": s.get("dilated_cloud_flag"),
                    "aggregate": s.get("aggregate_flag"), "valid": s.get("refl_pos"),
                    "wl_same": bool(len(wl) == len(wl_ref) and np.max(np.abs(wl - wl_ref)) < 1.0),
                    "ls_cc": cc, "profile": rw.profile, "fit": rw.fit, "kg": rw.kg, "scene_cloud_pct": rw.cloud_pct, "source": "extra"})
    except Exception as e:
        print("   facts failed:", rw.gee_index, str(e)[:100])
    if len(buf) >= 10 or (n == len(avail) and buf):
        pd.DataFrame(buf).to_csv(fp, mode="a", header=not os.path.exists(fp), index=False); n_new += len(buf); buf = []
    if n % 25 == 0: print(f"   facts {n}/{len(avail)}")
if buf: pd.DataFrame(buf).to_csv(fp, mode="a", header=not os.path.exists(fp), index=False); n_new += len(buf)
print(f"\n(3) facts computed this run: {n_new} | total in {os.path.basename(fp)}: {len(pd.read_csv(fp)) if os.path.exists(fp) else 0}")

# ---- 4) decisions ----
ex = pd.read_csv(fp).drop_duplicates("gee_index")
for c in ("sun_elev", "aod", "valid", "cloud", "ls_cc"): ex[c] = pd.to_numeric(ex[c], errors="coerce")
ex["decision"] = np.select(
    [~ex["wl_same"].astype(bool), ex["sun_elev"] < MIN_SUN, ex["aod"] > MAX_AOD, ex["valid"] < MIN_VALID, ex["cloud"] > MAX_CLOUD],
    ["skip_wavelengths", "skip_sun", "skip_aod", "skip_valid", "skip_cloud"], "process")
ex.to_csv(fp, index=False)
print("\n(4) decisions:", ex["decision"].value_counts().to_dict())
pr = ex[ex["decision"] == "process"]
print(f"    to process: {len(pr)} scenes in {pr['cell_id'].nunique()} cells | by profile: {pr['profile'].value_counts().to_dict()}")
print("    sun / AOD / Landsat cloud % (median):", round(pr["sun_elev"].median(), 1), round(pr["aod"].median(), 2), round(pr["ls_cc"].median(), 1))
print("    by year:", pr["date"].astype(str).str[:4].value_counts().sort_index().to_dict())

# ---- 5) append the 'process' rows to emit_cell_stats.csv (backup once) ----
sp = f"{BATCH}/emit_cell_stats.csv"
if not os.path.exists(f"{BATCH}/emit_cell_stats_v1.csv"): shutil.copy(sp, f"{BATCH}/emit_cell_stats_v1.csv")
st_all = pd.read_csv(sp)
add = pr[~pr["gee_index"].isin(set(st_all["gee_index"]))].drop(columns=["decision"])
if len(add):
    st_all = pd.concat([st_all, add], ignore_index=True)
    st_all["source"] = st_all["source"].fillna("handover") if "source" in st_all else "handover"
    st_all.to_csv(sp, index=False)
print(f"\n(5) appended to emit_cell_stats.csv: {len(add)} | rows now: {len(pd.read_csv(sp))}  (backup: emit_cell_stats_v1.csv)")

## A4 Earth Engine exports (submit + status)
One export per scene in `emit_cell_stats.csv` (handover and extra): EMIT reflectance (kept bands), flags, AOD/H2O/elevation and
Landsat 8/9 surface reflectance, LST and clear fraction, all on the **EMIT 60 m grid** of that scene (Landsat averaged over clear 30 m pixels).
Files land in `MEMBER1___PREPROCESSING/`; B1 moves them. Re-running submits nothing twice and prints the task states. (= old E-04 export + E-11)

In [ ]:
# A4. Submit one Earth Engine export per scene (handover + extra) and show task status. Resumable. (= old E-04 export + E-11)
import os, json
import numpy as np, pandas as pd, ee

# ---- idempotent setup ----
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"; ROOT = f"{BASE}/EMIT-LANDSAT"
PROC, PAIRS, BATCH, CONF = f"{ROOT}/processed", f"{ROOT}/pairs", f"{ROOT}/batch", f"{ROOT}/config"
for d in ("raw", "processed", "pairs", "batch", "config"): os.makedirs(f"{ROOT}/{d}", exist_ok=True)
EE_PROJECT, CH, HALF = "hyperspectral-thermal-511303", "NASA/EMIT/L2A/RFL", 0.035
try: ee.Initialize(project=EE_PROJECT)
except Exception: ee.Authenticate(); ee.Initialize(project=EE_PROJECT)
LSBANDS = ["blue", "green", "red", "nir", "swir1", "swir2", "LST_K", "clear_frac"]
MAX_SUBMIT = 400                     # tasks submitted in this run

exp = json.load(open(f"{CONF}/emit_export_bands.json"))
keep, FLAGS, EXTRA = exp["emit_reflectance_band_indices"], exp["emit_flags"], exp["emit_extra"]
assert exp["landsat"] == LSBANDS, "Landsat band list differs from the config"
st = pd.read_csv(f"{BATCH}/emit_cell_stats.csv").drop_duplicates("gee_index")
st["source"] = st["source"].fillna("handover") if "source" in st.columns else "handover"
print(f"scenes in emit_cell_stats.csv: {len(st)} | by source: {st['source'].value_counts().to_dict()}")

def export_pair(r):
    emit = ee.Image(f"{CH}/{r['gee_index']}")
    pj = emit.select("reflectance_50").projection().getInfo()
    crs, tf = pj["crs"], pj["transform"]
    sel = emit.select([f"reflectance_{k}" for k in keep] + FLAGS + EXTRA)
    lid = r["ls_id"]; coll = "LANDSAT/LC09/C02/T1_L2" if lid.startswith("LC09") else "LANDSAT/LC08/C02/T1_L2"
    L = ee.Image(f"{coll}/{lid}"); qa = L.select("QA_PIXEL")
    clear = qa.bitwiseAnd(0b11110).eq(0).And(qa.neq(0))                       # QA bits 1-4 (dilated cloud, cirrus, cloud, shadow) = 0
    sr = (L.select(["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"]).multiply(0.0000275).add(-0.2)
            .rename(LSBANDS[:6]))
    lst = L.select("ST_B10").multiply(0.00341802).add(149.0).rename("LST_K")
    lan = sr.addBands(lst).updateMask(clear).addBands(clear.rename("clear_frac").toFloat())
    lan = lan.reduceResolution(reducer=ee.Reducer.mean(), maxPixels=1024).reproject(crs=crs, crsTransform=tf)
    out = sel.addBands(lan).toFloat().unmask(-9999)
    reg = ee.Geometry.Rectangle([r["lon"] - HALF, r["lat"] - HALF, r["lon"] + HALF, r["lat"] + HALF], geodesic=False)
    name = f"emitpair_{r['gee_index']}"
    ee.batch.Export.image.toDrive(image=out, description=name[:100], folder="MEMBER1___PREPROCESSING", fileNamePrefix=name,
                                  region=reg, crs=crs, crsTransform=tf, maxPixels=1e9, fileFormat="GeoTIFF").start()
    return name

# ---- submit (skip anything already submitted or exported) ----
sp = f"{BATCH}/emit_export_state.csv"
state = pd.read_csv(sp) if os.path.exists(sp) else pd.DataFrame(columns=["name", "gee_index", "why", "ls_id"])
done = set(state["name"]); added, n_fail = [], 0
for _, r in st.iterrows():
    if len(added) >= MAX_SUBMIT: print("MAX_SUBMIT reached; run again for the rest"); break
    nm = f"emitpair_{r['gee_index']}"
    if nm in done or os.path.exists(f"{BASE}/{nm}.tif") or os.path.exists(f"{PROC}/{nm}.tif"): continue
    try:
        export_pair(r); added.append({"name": nm, "gee_index": r["gee_index"], "why": r["source"], "ls_id": r["ls_id"]})
    except Exception as e:
        n_fail += 1; print("   submit failed:", nm[9:60], "|", str(e)[:150])
        if any(k in str(e).lower() for k in ("quota", "too many", "rate")): print("   Earth Engine limit reached; run again later"); break
    if added and len(added) % 25 == 0:
        pd.concat([state, pd.DataFrame(added)], ignore_index=True).to_csv(sp, index=False); print(f"   submitted {len(added)}")
if added: state = pd.concat([state, pd.DataFrame(added)], ignore_index=True); state.to_csv(sp, index=False)
print(f"\nsubmitted this run: {len(added)} | failed: {n_fail} | total in export state: {len(state)}")

# ---- status of all emitpair tasks + files already on Drive ----
try:
    ops = [o for o in ee.data.listOperations() if str(o.get("metadata", {}).get("description", "")).startswith("emitpair_")]
    print("Earth Engine task states (emitpair_*):", pd.Series([o["metadata"].get("state") for o in ops], dtype=object).value_counts().to_dict())
    failed = [(o["metadata"]["description"][9:60], (o.get("error") or {}).get("message", "")[:80]) for o in ops if o["metadata"].get("state") == "FAILED"]
    if failed: print("FAILED tasks (first 10):", failed[:10])
except Exception as e: print("could not list tasks:", str(e)[:100])
want = {f"emitpair_{g}" for g in st["gee_index"]}
on_drive = {f[:-4] for f in os.listdir(BASE) if f.startswith("emitpair_") and f.endswith(".tif")} | \
           {f[:-4] for f in os.listdir(PROC) if f.startswith("emitpair_") and f.endswith(".tif")}
print(f"exports on Drive: {len(want & on_drive)} of {len(want)}")
print("Tasks: https://code.earthengine.google.com/tasks  |  run A4 again to check progress; run B1 when (almost) all are on Drive")

# PART B: pairs, QC and handover

## B1 QC of all exports -> pair folders (EMIT QC v2.3)
For every export in `processed/`:
1. **Valid EMIT pixel:** `cloud_flag == 0` and at least 80% of the kept bands > 0 (EMIT's dilated-cloud and aggregate flags are not used).
2. **Alignment:** whole-pixel shift search +-8 px on NDVI; a shift is applied only if the correlation clearly improves (best r > 0.6, gain > 0.03).
   A shift > 3 px must be confirmed by >= 2 of 3 independent signals (red, SWIR1, brightness) peaking within 1 px.
3. **Residual cloud / shadow mask** (after alignment): EMIT pixels much brighter in blue (cloud) or much darker in NIR (shadow) than Landsat,
   relative to the pair's own median ratio (5x5 median difference > 0.06 and factor 1.5), buffered 2 px; where Landsat is not clear, EMIT blue > 0.35.
   Masked pixels become nodata in the delivered EMIT file.
4. **Checks (all must pass):** joint coverage >= 0.5, NIR ratio 0.85-1.15, NDVI r >= 0.7, AOD <= 0.8, sun >= 25 deg, Landsat clear >= 0.85,
   no unresolved shift, large shift confirmed, **NDVI gain >= 0.80** (haze; gain = mean EMIT NDVI / mean Landsat NDVI over Landsat NDVI > 0.3).
5. **Flags only:** blue / red / SWIR1 ratio outside 0.85-1.15; `haze_mild` (gain 0.80-0.85); `haze_unchecked` (< 500 vegetated pixels).
6. Writes `pairs/<N>/` (pair numbers append-only), the Koppen code and climate group, and `pairs/emit_final_qc_v2.3.csv`.

Safe to re-run: the pair files are rebuilt from the raw exports, so the same input gives the same result. Run time about 10-20 min for 400 exports.

In [ ]:
# B1. EMIT QC v2.3 for ALL EMIT exports (handover + extra) -> pair folders. Safe to re-run. (= old E-13)
#       v2.1 (E-08 rules + new pair numbers + large-shift confirmation + Koppen)
#       + residual cloud / shadow mask (E-12c) + scene haze rule on NDVI gain (E-12d)
#       v2.3 = v2.2 with haze threshold 0.80 (was 0.75) and a 2 px cloud/shadow buffer (was 1 px), after the E-13 figure (archive notebook)
import os, re, glob, json, shutil, warnings
import numpy as np, pandas as pd, rasterio
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter, binary_dilation
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---- idempotent setup ----
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"; ROOT = f"{BASE}/EMIT-LANDSAT"
PROC, PAIRS = f"{ROOT}/processed", f"{ROOT}/pairs"
for d in (PROC, PAIRS): os.makedirs(d, exist_ok=True)

# ---- EMIT QC v2.3 rules (design choices; edit here) ----
QC_VERSION = "v2.3"                                             # CHANGED v2.3
MIN_COV, RLO, RHI = 0.5, 0.85, 1.15
MIN_NDVI_R, MAX_AOD, MIN_SUN, MIN_LS_CLEAR = 0.7, 0.8, 25.0, 0.85
MIN_POS_BANDS, SWIR1_HARD = 0.8, False
SHIFT_MAX, MIN_R_SHIFT, MIN_GAIN, BIG_SHIFT = 8, 0.6, 0.03, 3
CONFIRM_MIN = 2                                                 # large shift needs >= 2 of 3 independent signals (red, swir1, bright)
FLAG_BANDS = ("blue", "red", "swir1")
# residual cloud / shadow mask (relative to the pair's own median EMIT/Landsat ratio; EMIT dilated_cloud_flag NOT used)
K_RATIO, D_REF, MIN_LS_NIR, ABS_BLUE = 1.5, 0.06, 0.10, 0.35
DILATE_PX = 2                                                   # CHANGED v2.3 (was 1): thin cloud edges slipped through (pair 248)
# scene haze rule: NDVI gain = mean EMIT NDVI / mean Landsat NDVI over Landsat NDVI > 0.3 (after the mask)
VEG_NDVI, MIN_NDVI_GAIN, MILD_NDVI_GAIN, MIN_VEG_PX = 0.3, 0.80, 0.85, 500   # CHANGED v2.3: 0.80 (was 0.75; pairs 254/250/323 hazy at 0.76)
WIN = {"blue": (452, 512), "green": (533, 590), "red": (636, 673),
       "nir": (851, 879), "swir1": (1566, 1651), "swir2": (2107, 2294)}
NAMES = list(WIN)
fx = lambda x, d=3: None if x is None or not np.isfinite(x) else round(float(x), d)
pd.set_option("display.width", 260)

# ---- config + metadata ----
cfg = json.load(open(f"{ROOT}/config/emit_bands.json")); exp = json.load(open(f"{ROOT}/config/emit_export_bands.json"))
wl_all, keep = np.array(cfg["wavelength_nm"]), cfg["keep_band_indices"]; wlk = wl_all[keep]
FLAGS, EXTRA, LSN = exp["emit_flags"], exp["emit_extra"], exp["landsat"]
nk, nf, nx = len(keep), len(FLAGS), len(EXTRA); NEXP = nk + nf + nx + len(LSN)
stats = pd.read_csv(f"{ROOT}/batch/emit_cell_stats.csv").drop_duplicates("gee_index").set_index("gee_index")
ipath = f"{PAIRS}/pair_index.csv"
idx = pd.read_csv(ipath) if os.path.exists(ipath) else pd.DataFrame(columns=["pair_no", "scene_id", "cell_id", "profile", "landsat_id"])
ho = pd.read_csv(f"{BASE}/handover_pairs.csv"); ho = ho[ho["sensor"] == "EMIT"]
stamp = lambda s: (lambda m_: m_.group(1) if m_ else None)(re.search(r"(\d{8}T\d{6})", str(s)))
kg_of = {stamp(s): k for s, k in zip(ho["scene_id"], ho["kg"])}       # keyed by acquisition timestamp
def cgroup(k):
    if k is None or pd.isna(k): return None
    k = int(k); return "tropical" if k <= 3 else "arid" if k <= 7 else "temperate" if k <= 16 else "cold" if k <= 28 else "polar"

# ---- helpers ----
def sh(A, dy, dx): return A[max(dy, 0):A.shape[0] + min(dy, 0), max(dx, 0):A.shape[1] + min(dx, 0)]
def pearson(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.corrcoef(a[m], b[m])[0, 1]) if m.sum() >= 200 else np.nan
def shift_cube(A, dy, dx):
    out = np.full_like(A, np.nan); hh, ww = A.shape[-2:]
    ys, ye, xs, xe = max(0, -dy), min(hh, hh - dy), max(0, -dx), min(ww, ww - dx)
    out[..., ys:ye, xs:xe] = A[..., ys + dy:ye + dy, xs + dx:xe + dx]; return out
def win_mean(H, k):
    sel = (wlk >= WIN[k][0]) & (wlk <= WIN[k][1])
    return np.nanmean(H[sel], axis=0) if sel.any() else np.full(H.shape[1:], np.nan)
def ndvi_pair(H, SR, clear):
    rd, ni = win_mean(H, "red"), win_mean(H, "nir")
    nh = (ni - rd) / (ni + rd); nl = (SR[3] - SR[2]) / (SR[3] + SR[2])
    u = (np.isfinite(nh) & np.isfinite(nl) & clear & (rd > 0) & (ni > 0) & (SR[2] > 0) & (SR[3] > 0) & (np.abs(nl) <= 1))
    return np.where(u, nh, np.nan), np.where(u, nl, np.nan)
def best_shift(nh, nl):
    r0 = pearson(nh, nl); best = (r0, (0, 0))
    for dy in range(-SHIFT_MAX, SHIFT_MAX + 1):
        for dx in range(-SHIFT_MAX, SHIFT_MAX + 1):
            r = pearson(sh(nh, dy, dx), sh(nl, -dy, -dx))
            if np.isfinite(r) and (not np.isfinite(best[0]) or r > best[0]): best = (r, (dy, dx))
    return r0, best
def confirm(H, SR, clear):                                       # E-08b as a rule (on the shifted cube)
    sig = {"red": (win_mean(H, "red"), SR[2]), "swir1": (win_mean(H, "swir1"), SR[4]),
           "bright": (np.nanmean([win_mean(H, k) for k in NAMES], axis=0), np.nanmean(SR, axis=0))}
    det, n_ok = {}, 0
    for k, (a, b) in sig.items():
        a = np.where(clear & np.isfinite(a), a, np.nan); b = np.where(clear & np.isfinite(b) & (b > 0), b, np.nan)
        _, (rb, s) = best_shift(a, b); ok = max(abs(s[0]), abs(s[1])) <= 1
        det[k] = [int(s[0]), int(s[1])]; n_ok += ok
    return n_ok, det
def med5(a): return median_filter(np.where(np.isfinite(a), a, 0).astype("float32"), size=5)
def mratio(a, b, m):
    m = m & np.isfinite(a) & np.isfinite(b) & (b > 0)
    return float(np.median(a[m]) / np.median(b[m])) if m.sum() >= 1000 else np.nan
def cloud_mask(H, SR, clear):                                    # E-12c rule
    fin = np.isfinite(H).any(axis=0); joint = fin & clear
    Eb, En = win_mean(H, "blue"), win_mean(H, "nir")
    kb, kn = mratio(Eb, SR[0], joint), mratio(En, SR[3], joint)
    absc = fin & ~clear & (Eb > ABS_BLUE)
    if not (np.isfinite(kb) and np.isfinite(kn)): z = np.zeros_like(fin); return z, z, absc
    bright = joint & (med5(Eb - kb * SR[0]) > D_REF) & (Eb > K_RATIO * kb * SR[0])
    shadow = joint & (SR[3] > MIN_LS_NIR) & (med5(En - kn * SR[3]) < -D_REF) & (En < kn * SR[3] / K_RATIO)
    return (binary_dilation(bright, iterations=DILATE_PX) & fin, binary_dilation(shadow, iterations=DILATE_PX) & fin,   # CHANGED v2.3
            binary_dilation(absc, iterations=DILATE_PX) & fin)
def ndvi_gain(nh, nl):                                           # E-12d metric
    v = np.isfinite(nh) & np.isfinite(nl) & (nl > VEG_NDVI)
    return float(np.mean(nh[v]) / np.mean(nl[v])) if v.sum() >= MIN_VEG_PX else np.nan

# ---- move new exports from the Drive top level into processed/ ----
for f in glob.glob(f"{BASE}/emitpair_*.tif"):
    dst = f"{PROC}/{os.path.basename(f)}"
    if not os.path.exists(dst): shutil.move(f, dst)
files = {os.path.basename(f)[:-4]: f for f in glob.glob(f"{PROC}/emitpair_*.tif")}
sp = f"{ROOT}/batch/emit_export_state.csv"
missing = [n for n in (pd.read_csv(sp)["name"] if os.path.exists(sp) else []) if n not in files]
print(f"exports in processed/: {len(files)} | submitted but not on Drive yet: {len(missing)}")

rows, viz = [], {}
for n, (name, path) in enumerate(sorted(files.items()), 1):
    gi = name.replace("emitpair_", "")
    if gi not in stats.index: rows.append({"pair": None, "scene": gi[17:32], "status": "FAIL", "reasons": "no metadata"}); continue
    m = stats.loc[gi]
    with rasterio.open(path) as src: arr, tf, crs = src.read().astype("float32"), src.transform, src.crs
    if arr.shape[0] != NEXP: rows.append({"pair": None, "scene": gi[17:32], "status": "FAIL", "reasons": f"bands {arr.shape[0]} != {NEXP}"}); continue
    arr[arr == -9999] = np.nan
    R, F, X, L = arr[:nk], arr[nk:nk + nf], arr[nk + nf:nk + nf + nx], arr[nk + nf + nx:]
    valid = (F[FLAGS.index("cloud_flag")] == 0) & ((R > 0).mean(axis=0) >= MIN_POS_BANDS)
    H = R.copy(); H[:, ~valid] = np.nan
    SR, cf = L[:6], L[7]
    ls_has = np.isfinite(cf)
    clear = ls_has & np.isfinite(SR[3]) & (cf >= 0.8)

    # registration (+-8 px)
    nh, nl = ndvi_pair(H, SR, clear); r0, (rb, (dy, dx)) = best_shift(nh, nl)
    if (dy, dx) != (0, 0) and np.isfinite(rb) and rb > MIN_R_SHIFT and (rb - r0) > MIN_GAIN: H = shift_cube(H, dy, dx)
    else: dy, dx = 0, 0

    # residual cloud / shadow mask on the aligned cube; everything below is computed after masking
    fin0 = np.isfinite(H).any(axis=0); nj = max(1, int((fin0 & clear).sum()))
    b_m, s_m, a_m = cloud_mask(H, SR, clear)
    H[:, b_m | s_m | a_m] = np.nan
    cfr = {"bright_cloud": fx((b_m & clear).sum() / nj), "cloud_shadow": fx((s_m & clear).sum() / nj),
           "bright_where_landsat_not_clear": fx(a_m.sum() / max(1, int(fin0.sum())))}

    nh2, nl2 = ndvi_pair(H, SR, clear); r_after, (r_res, sh_res) = best_shift(nh2, nl2)
    at_edge = SHIFT_MAX in (abs(dy), abs(dx))
    unresolved = at_edge or (sh_res != (0, 0) and np.isfinite(r_res) and (r_res - r_after) > MIN_GAIN)
    big = max(abs(dy), abs(dx)) > BIG_SHIFT
    n_conf, conf_det = confirm(H, SR, clear) if big else (None, None)
    big_ok = (not big) or (n_conf >= CONFIRM_MIN)
    gain = ndvi_gain(nh2, nl2)
    mild = bool(np.isfinite(gain) and MIN_NDVI_GAIN <= gain < MILD_NDVI_GAIN)

    # coverage (clear fraction only where Landsat has data; joint = EMIT valid AND Landsat clear)
    fin = np.isfinite(H).any(axis=0)
    cov_emit = float(fin.mean())
    ls_cover = float(ls_has[fin].mean()) if fin.any() else np.nan
    ls_clear = float((cf[fin & ls_has] >= 0.8).mean()) if (fin & ls_has).any() else np.nan
    cov = float((fin & clear).mean())
    ratio = {}
    for i, k in enumerate(NAMES):
        a_ = win_mean(H, k); b_ = SR[i]; mm = np.isfinite(a_) & clear & (b_ > 0)
        ratio[k] = float(np.median(a_[mm]) / np.median(b_[mm])) if mm.sum() >= 1000 else np.nan
    flag_ok = {k: bool(RLO <= ratio[k] <= RHI) for k in FLAG_BANDS}
    sun, aod = float(m["sun_elev"]), float(m["aod"])
    checks = {"coverage": cov >= MIN_COV, "nir_ratio": bool(RLO <= ratio["nir"] <= RHI),
              "swir1_ratio": flag_ok["swir1"] if SWIR1_HARD else True, "no_unresolved_shift": not unresolved,
              "ndvi_r": bool(np.isfinite(r_after) and r_after >= MIN_NDVI_R), "aod": aod <= MAX_AOD,
              "sun_elev": sun >= MIN_SUN, "landsat_clear": bool(np.isfinite(ls_clear) and ls_clear >= MIN_LS_CLEAR),
              "large_shift_confirmed": bool(big_ok),
              "haze_ndvi_gain": bool(not np.isfinite(gain) or gain >= MIN_NDVI_GAIN)}
    status = "PASS" if all(checks.values()) else "FAIL"; reasons = ",".join(k for k, v in checks.items() if not v)
    flags_out = ",".join([k for k, v in flag_ok.items() if not v] + (["haze_mild"] if mild else [])
                         + (["haze_unchecked"] if not np.isfinite(gain) else []))

    # pair number (append-only) + folder
    hit = idx[idx["scene_id"] == gi]
    prof = m["profile"] if "profile" in m.index and pd.notna(m["profile"]) else "humid_coastal"
    if len(hit): no = int(hit.iloc[0]["pair_no"])
    else:
        no = int(idx["pair_no"].max()) + 1 if len(idx) else 1
        idx.loc[len(idx)] = [no, gi, m["cell_id"], prof, m["ls_id"]]; idx.to_csv(ipath, index=False)
    folder = f"{PAIRS}/{no}"; os.makedirs(folder, exist_ok=True); jp = f"{folder}/pair_info.json"
    prev_status = json.load(open(jp))["final_qc"]["status"] if os.path.exists(jp) else None
    if os.path.exists(jp) and not os.path.exists(f"{folder}/pair_info_v1.json"): shutil.copy(jp, f"{folder}/pair_info_v1.json")

    out = np.full(H.shape, -32768, "int16"); ok = np.isfinite(H); out[ok] = np.clip(np.round(H[ok] * 10000), -32000, 32000).astype("int16")
    with rasterio.open(f"{folder}/emit_reflectance.tif", "w", driver="GTiff", height=out.shape[1], width=out.shape[2], count=nk,
                       dtype="int16", crs=crs, transform=tf, nodata=-32768, compress="deflate") as dst:
        dst.write(out)
        for k, (b, w) in enumerate(zip(keep, wlk), start=1): dst.set_band_description(k, f"{w:.0f} nm (EMIT b{int(b):03d})")
        dst.update_tags(scale_factor_inverse=10000, units="surface reflectance x 10000", source=gi, product="EMIT L2A (GEE NASA/EMIT/L2A/RFL)",
                        valid_pixel="cloud_flag==0 and >=80% kept bands >0, minus residual bright cloud / cloud shadow (relative to Landsat) "
                                    f"and EMIT blue > 0.35 where Landsat is not clear, buffered {DILATE_PX} px",
                        shift_applied_yx=str((dy, dx)), qc_version=QC_VERSION)
    with rasterio.open(f"{folder}/landsat_{m['ls_id']}.tif", "w", driver="GTiff", height=L.shape[1], width=L.shape[2], count=L.shape[0],
                       dtype="float32", crs=crs, transform=tf, nodata=np.nan, compress="deflate") as dst:
        dst.write(L.astype("float32"))
        for k, nme in enumerate(LSN, start=1): dst.set_band_description(k, nme)
        dst.update_tags(note="Landsat averaged onto the EMIT 60 m grid over clear pixels; clear_frac = fraction of clear 30 m pixels; NaN = no Landsat data")
    kg = m["kg"] if "kg" in m.index and pd.notna(m["kg"]) else kg_of.get(stamp(gi))
    source = m["source"] if "source" in m.index and pd.notna(m["source"]) else "handover"
    info = {"pair_no": no, "emit_scene": gi, "emit_date": str(m["date"]), "landsat_scene": m["ls_id"], "landsat_minus_emit_days": fx(float(m["ls_dt"]), 4),
            "cell_id": m["cell_id"], "cell_lon": float(m["lon"]), "cell_lat": float(m["lat"]), "profile": prof, "source_list": source,
            "koppen_code": int(kg) if kg is not None and pd.notna(kg) else None, "climate_group": cgroup(kg),
            "status": status, "qc_version": QC_VERSION,
            "atmosphere": {"aod550": aod, "water_vapour_gcm2": fx(float(m["h2o"]), 2), "elev_m": fx(float(m["elev"]), 1)},
            "final_qc": {"status": status, "failed_checks": reasons, "flags_outside_range": flags_out, "swir1_flag_ok": flag_ok["swir1"],
                         "coverage_joint": fx(cov), "coverage_emit": fx(cov_emit), "landsat_cover_of_emit": fx(ls_cover),
                         "landsat_clear_frac": fx(ls_clear), "sun_elev": fx(sun, 1), "aod550": fx(aod, 2),
                         "ndvi_r_before": fx(r0), "ndvi_r_after": fx(r_after), "ndvi_gain": fx(gain), "haze_mild": mild,
                         "residual_cloud_masked_frac": cfr,
                         "applied_shift_yx": [dy, dx], "large_shift": big,
                         "large_shift_confirming_signals": n_conf, "large_shift_signal_peaks": conf_det,
                         "shift_at_search_edge": at_edge, "ratio_emit_over_landsat": {k: fx(v, 2) for k, v in ratio.items()},
                         "flag_fractions": {fl: fx(np.nanmean(F[i]), 3) for i, fl in enumerate(FLAGS)}, "n_bands": nk,
                         "dropped_band_indices": cfg["dropped_band_indices"],
                         "thresholds": {"min_cov_joint": MIN_COV, "ratio": [RLO, RHI], "max_aod": MAX_AOD, "min_sun": MIN_SUN,
                                        "min_ndvi_r": MIN_NDVI_R, "min_landsat_clear": MIN_LS_CLEAR, "swir1_hard": SWIR1_HARD,
                                        "shift_search_px": SHIFT_MAX, "shift_min_r": MIN_R_SHIFT, "shift_min_gain": MIN_GAIN,
                                        "large_shift_px": BIG_SHIFT, "large_shift_min_confirming": CONFIRM_MIN,
                                        "cloud_mask": {"ratio_factor": K_RATIO, "min_diff": D_REF, "shadow_min_landsat_nir": MIN_LS_NIR,
                                                       "abs_blue_where_landsat_not_clear": ABS_BLUE, "median_window": 5, "dilation_px": DILATE_PX},
                                        "haze": {"veg_landsat_ndvi": VEG_NDVI, "min_ndvi_gain": MIN_NDVI_GAIN,
                                                 "mild_below": MILD_NDVI_GAIN, "min_veg_px": MIN_VEG_PX},
                                        "flag_bands": list(FLAG_BANDS)}}}
    json.dump(info, open(jp, "w"), indent=1, default=str)

    rows.append({"pair": no, "source": source, "date": m["date"], "cell": m["cell_id"], "climate": cgroup(kg), "sun": fx(sun, 1),
                 "cov": fx(cov, 2), "ls_clear": fx(ls_clear, 2), "cloud%": fx(100 * ((cfr["bright_cloud"] or 0) + (cfr["cloud_shadow"] or 0)), 1),
                 "gain": fx(gain, 2), "shift": f"{dy},{dx}", "confirm": n_conf,
                 "ndvi_r": f"{fx(r0, 2)}->{fx(r_after, 2)}", **{k: fx(ratio[k], 2) for k in NAMES},
                 "prev": prev_status, "status": status, "reasons": reasons, "flags": flags_out, "big": big, "mild": mild})
    if (big and not big_ok) or (mild and status == "PASS"):      # CHANGED v2.3: confirmed shifts already checked by eye
        ib = [int(np.argmin(np.abs(wlk - w))) for w in (650, 550, 470)]
        viz[no] = (np.nan_to_num(np.clip(np.stack([H[i] for i in ib], -1) / 0.3, 0, 1)),
                   np.nan_to_num(np.clip(np.stack([SR[2], SR[1], SR[0]], -1) / 0.3, 0, 1)), nh2, nl2, status, (dy, dx), n_conf, gain)
    if n % 50 == 0: print(f"  {n}/{len(files)} exports")

# ---- summary ----
res = pd.DataFrame(rows).sort_values("pair").reset_index(drop=True)
res.to_csv(f"{PAIRS}/emit_final_qc_{QC_VERSION}.csv", index=False)
ok_rows = res[res["pair"].notna()]
print(f"\nQC {QC_VERSION} | pairs: {len(ok_rows)} | PASS: {int((ok_rows['status'] == 'PASS').sum())} | FAIL: {int((ok_rows['status'] == 'FAIL').sum())}"
      f" | no metadata / wrong bands: {int(res['pair'].isna().sum())}")
print("by source:\n", pd.crosstab(ok_rows["source"], ok_rows["status"]).to_string())
old = ok_rows[ok_rows["prev"].notna()]
chg = old[old["prev"] != old["status"]]
print(f"\npairs whose status changed vs the previous run: {len(chg)}")
if len(chg): print(chg[["pair", "source", "date", "cell", "cov", "cloud%", "gain", "ndvi_r", "prev", "status", "reasons"]].to_string(index=False))
pp = ok_rows[ok_rows["status"] == "PASS"]
print(f"\nPASS cells: {pp['cell'].nunique()} | PASS pairs per cell: {pp['cell'].value_counts().value_counts().sort_index().to_dict()}")
print("PASS by climate group:", pp["climate"].value_counts(dropna=False).to_dict())
fails = ok_rows[ok_rows["status"] == "FAIL"].assign(reason=lambda x: x["reasons"].str.split(",")).explode("reason")
print("\nfailed checks (a pair can fail several):", fails["reason"].value_counts().to_dict())
print("failing ONE check only:", ok_rows[(ok_rows["status"] == "FAIL") & ~ok_rows["reasons"].str.contains(",")]["reasons"].value_counts().to_dict())
print(f"\nresidual cloud/shadow masked (% of joint px) among PASS: median {pp['cloud%'].median()} | > 5%: {int((pp['cloud%'] > 5).sum())} | > 20%: {int((pp['cloud%'] > 20).sum())}")
print(f"NDVI gain among PASS: quantiles 5/25/50/75 % = {pp['gain'].quantile([.05, .25, .5, .75]).round(2).tolist()}"
      f" | haze_mild PASS: {int(pp['mild'].sum())} | haze_unchecked PASS: {int(pp['flags'].fillna('').str.contains('haze_unchecked').sum())}")
bg = ok_rows[ok_rows["big"] == True]
print(f"\nLARGE SHIFTS (> {BIG_SHIFT} px): {len(bg)} | confirmed (>= {CONFIRM_MIN}/3): {int((bg['confirm'] >= CONFIRM_MIN).sum())}")
if len(bg): print(bg[["pair", "source", "date", "cell", "shift", "confirm", "ndvi_r", "status"]].to_string(index=False))
print("\nflags among PASS pairs:", {k: int(pp["flags"].fillna("").str.contains(k).sum()) for k in FLAG_BANDS + ("haze_mild",)})
print("median ratios over PASS pairs:", pp[NAMES].median().round(2).to_dict())
nfail = ok_rows[ok_rows["reasons"].str.contains("nir_ratio")]
print(f"NIR-ratio failures: {len(nfail)} | too low: {int((nfail['nir'] < RLO).sum())} | too high: {int((nfail['nir'] > RHI).sum())}")

# ---- visual check (max 8): unconfirmed large shifts first, then haze_mild PASS pairs (lowest gain first) ----
shifts = [p for p in sorted(viz) if res.loc[res["pair"] == p, "big"].iloc[0]]
mildp = [p for p in res[res["mild"] & (res["status"] == "PASS")].sort_values("gain")["pair"] if p in viz and p not in shifts]
show = (shifts + mildp)[:8]
if show:
    fig, axs = plt.subplots(len(show), 3, figsize=(12, 3.6 * len(show)), squeeze=False)
    for i, no in enumerate(show):
        ae, al, a2, b2, stt, s, nc, g = viz[no]
        ttl = f"pair {no} shift {s} confirmed {nc}/3 {stt}" if no in shifts else f"pair {no} haze_mild gain {fx(g, 2)} {stt}"
        axs[i, 0].imshow(ae); axs[i, 0].set_title(ttl, fontsize=9)
        axs[i, 1].imshow(al); axs[i, 1].set_title("Landsat RGB (60 m average)", fontsize=9)
        mk = np.isfinite(a2) & np.isfinite(b2); sel = np.flatnonzero(mk.ravel())[::max(1, mk.sum() // 3000)]
        axs[i, 2].scatter(b2.ravel()[sel], a2.ravel()[sel], s=2, alpha=.4); axs[i, 2].plot([0, 1], [0, 1], "k--", lw=.8)
        axs[i, 2].set_xlim(-0.2, 1); axs[i, 2].set_ylim(-0.2, 1); axs[i, 2].set_xlabel("Landsat NDVI"); axs[i, 2].set_ylabel("EMIT NDVI")
        for a in axs[i, :2]: a.axis("off")
    plt.tight_layout(); plt.show()
print(f"\nsaved: {PAIRS}/emit_final_qc_{QC_VERSION}.csv | waiting for {len(missing)} exports (re-run B1 when they arrive, then B2)")

## B2 Handover: copy PASS pairs to PASSED-PAIRS
`pairs/` is the master archive (all pairs, PASS and FAIL). **Member 2 only receives `PASSED-PAIRS/`**, so this cell has to run after every B1 run:
it copies new or changed PASS pairs, **moves** pairs that no longer pass to `PASSED-PAIRS/_to_delete/` (never deletes), and writes
`handover_pass.csv`, `emit_bands.json` and `README.txt`. Pairs listed in `EXCLUDE` are left out. (= old E-09, updated for QC v2.3)

In [ ]:
# B2. Copy EMIT PASS pairs into EMIT-LANDSAT/PASSED-PAIRS (+ handover_pass.csv + README). Safe to re-run. (= old E-09, updated)
import os, glob, json, shutil, time
import pandas as pd

# ---- idempotent setup ----
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"; ROOT = f"{BASE}/EMIT-LANDSAT"
PROC, PAIRS, BATCH, CONF = f"{ROOT}/processed", f"{ROOT}/pairs", f"{ROOT}/batch", f"{ROOT}/config"
for d in ("raw", "processed", "pairs", "batch", "config"): os.makedirs(f"{ROOT}/{d}", exist_ok=True)
OUT = f"{ROOT}/PASSED-PAIRS"; os.makedirs(OUT, exist_ok=True)
EXCLUDE = []                      # pair numbers to leave out after a visual check (e.g. [27])
DELIVER = ("emit_reflectance.tif", "landsat_*.tif", "pair_info.json")   # pair_info_v1.json stays in pairs/ only

# ---- 1) status of every pair folder ----
pidx = pd.read_csv(f"{PAIRS}/pair_index.csv").set_index("pair_no")
folders = sorted([d for d in glob.glob(f"{PAIRS}/*") if os.path.isdir(d) and os.path.basename(d).isdigit()],
                 key=lambda d: int(os.path.basename(d)))
rows, problems = [], []
for d in folders:
    no = int(os.path.basename(d)); jp = f"{d}/pair_info.json"
    if not os.path.exists(jp): problems.append(f"pair {no}: no pair_info.json"); continue
    info = json.load(open(jp)); q = info.get("final_qc") or {}; r = q.get("ratio_emit_over_landsat") or {}
    fl = q.get("flags_outside_range") or ""; cm = q.get("residual_cloud_masked_frac") or {}
    rows.append({"pair": no, "status": info.get("status"), "qc_version": info.get("qc_version"), "failed_checks": q.get("failed_checks"),
                 "profile": pidx["profile"].get(no), "source": info.get("source_list"),
                 "koppen_code": info.get("koppen_code"), "climate_group": info.get("climate_group"),
                 "cell_id": info.get("cell_id"), "cell_lon": info.get("cell_lon"), "cell_lat": info.get("cell_lat"),
                 "emit_scene": info.get("emit_scene"), "emit_date": info.get("emit_date"),
                 "landsat_scene": info.get("landsat_scene"), "landsat_minus_emit_days": info.get("landsat_minus_emit_days"),
                 "coverage_joint": q.get("coverage_joint"), "landsat_clear_frac": q.get("landsat_clear_frac"),
                 "sun_elev": q.get("sun_elev"), "aod550": q.get("aod550"), "ndvi_r_after": q.get("ndvi_r_after"),
                 "ndvi_gain": q.get("ndvi_gain"), "haze_mild": bool(q.get("haze_mild") or False),
                 "cloud_masked_frac": (cm.get("bright_cloud") or 0) + (cm.get("cloud_shadow") or 0),
                 "applied_shift_yx": q.get("applied_shift_yx"), "large_shift": q.get("large_shift"),
                 **{f"ratio_{k}": r.get(k) for k in ("blue", "green", "red", "nir", "swir1", "swir2")},
                 "blue_ok": "blue" not in fl, "red_ok": "red" not in fl, "swir1_ok": "swir1" not in fl,
                 "n_bands": q.get("n_bands")})
allp = pd.DataFrame(rows)
allp.to_csv(f"{PAIRS}/all_pairs_status.csv", index=False)
print("pairs found:", len(allp), "|", allp["status"].value_counts(dropna=False).to_dict(),
      "| qc versions:", allp["qc_version"].value_counts(dropna=False).to_dict())
if allp["qc_version"].nunique() > 1: problems.append("pairs have different QC versions -> run B1 first, then B2")

# ---- 2) PASS list minus EXCLUDE, with a completeness check ----
passp = allp[(allp["status"] == "PASS") & ~allp["pair"].isin(EXCLUDE)].copy()
ok_rows = []
for no in passp["pair"]:
    d = f"{PAIRS}/{no}"
    if all(glob.glob(f"{d}/{pat}") for pat in DELIVER): ok_rows.append(no)
    else: problems.append(f"pair {no}: PASS but a file is missing -> not copied")
passp = passp[passp["pair"].isin(ok_rows)]
keep = set(passp["pair"].astype(int))

# ---- 3) sync: stale copies go to _to_delete (never deleted), new/changed files copied ----
def copy_if_changed(src, dst):
    if (not os.path.exists(dst)) or os.path.getsize(dst) != os.path.getsize(src) or os.path.getmtime(dst) < os.path.getmtime(src):
        shutil.copy2(src, dst); return True
    return False
moved = []
for d in glob.glob(f"{OUT}/*"):
    if os.path.isdir(d) and os.path.basename(d).isdigit() and int(os.path.basename(d)) not in keep:
        os.makedirs(f"{OUT}/_to_delete", exist_ok=True); dst = f"{OUT}/_to_delete/{os.path.basename(d)}"
        if os.path.exists(dst): dst += "_" + time.strftime("%Y%m%d_%H%M%S")   # keep an older stale copy too; nothing is deleted
        shutil.move(d, dst); moved.append(int(os.path.basename(d)))
if moved: print(f"moved to _to_delete (no longer PASS): {len(moved)} pairs: {sorted(moved)}")
n_new = 0
for no in sorted(keep):
    src, dst = f"{PAIRS}/{no}", f"{OUT}/{no}"; os.makedirs(dst, exist_ok=True)
    files = sorted({os.path.basename(f) for pat in DELIVER for f in glob.glob(f"{src}/{pat}")})
    for f in files: n_new += copy_if_changed(f"{src}/{f}", f"{dst}/{f}")
    extra = [f for f in os.listdir(dst) if f not in files and f != "aux_layers.tif"]   # aux_layers.tif is made by FINAL_DATASET.ipynb A1
    if extra: problems.append(f"pair {no}: extra files in the copy, not touched: {extra}")
print(f"PASS pairs in PASSED-PAIRS: {len(keep)} ({n_new} files written or updated)")

# ---- 4) CSV, band config, README ----
passp.to_csv(f"{OUT}/handover_pass.csv", index=False)
shutil.copy2(f"{CONF}/emit_bands.json", f"{OUT}/emit_bands.json")
th = json.load(open(f"{PAIRS}/{sorted(keep)[0]}/pair_info.json"))["final_qc"]["thresholds"] if keep else {}
cm, hz = th.get("cloud_mask", {}), th.get("haze", {})
qcv = ", ".join(sorted(passp["qc_version"].dropna().astype(str).unique()))
n_cells = passp["cell_id"].nunique(); big = passp.loc[passp["large_shift"] == True, "pair"].tolist()
nb, nr, ns = int((~passp["blue_ok"]).sum()), int((~passp["red_ok"]).sum()), int((~passp["swir1_ok"]).sum())
nmild = int(passp["haze_mild"].sum())
readme = f"""PASSED-PAIRS: EMIT + Landsat 8/9 pairs that passed the quality rules (EMIT QC {qcv})
=========================================================================
Pairs: {len(passp)} in {n_cells} cells (pair numbers as in the master archive EMIT-LANDSAT/pairs/; gaps = not passed{', excluded: ' + str(EXCLUDE) if EXCLUDE else ''})
Climate groups (Koppen, Beck et al.): {passp['climate_group'].value_counts(dropna=False).to_dict()}

Each pair folder contains
  emit_reflectance.tif   EMIT L2A surface reflectance x 10000 (int16), nodata -32768 (masked or no data),
                         {passp['n_bands'].iloc[0] if len(passp) else ''} bands, wavelength in each band description
  landsat_<id>.tif       Landsat 8/9 on the SAME 60 m EMIT grid (float32, NaN = no data):
                         blue, green, red, nir, swir1, swir2 (surface reflectance), LST_K (kelvin), clear_frac (share of clear 30 m pixels)
  pair_info.json         scene IDs, dates, atmosphere values, Koppen code / climate group, QC numbers (final_qc)
handover_pass.csv lists every pair with cell, dates, climate and QC numbers. emit_bands.json is the band configuration.

How the pairs were made
  - EMIT L2A reflectance from Earth Engine (NASA/EMIT/L2A/RFL), used as delivered (no extra atmospheric correction)
  - water-vapour gap bands 127-141 and 187-212 dropped with a 3-band margin (232 of 285 kept)
  - EMIT pixel valid if cloud_flag == 0 and >= 80% of kept bands > 0
  - residual cloud and cloud shadow masked by comparison with Landsat (5x5 median difference > {cm.get('min_diff')} and factor
    {cm.get('ratio_factor')}, relative to the pair's own median EMIT/Landsat ratio, buffered {cm.get('dilation_px')} px);
    where Landsat is not clear, EMIT blue > {cm.get('abs_blue_where_landsat_not_clear')} masked
  - Landsat averaged onto the EMIT 60 m grid over clear pixels
  - EMIT shifted by whole pixels only where the NDVI correlation with Landsat clearly improved (search +-{th.get('shift_search_px')} px;
    applied_shift_yx); shifts > {th.get('large_shift_px')} px kept only if >= {th.get('large_shift_min_confirming')} of 3 independent signals agree

PASS rules (my design choices, not from the literature)
  joint coverage (EMIT valid and Landsat clear) >= {th.get('min_cov_joint')}, NIR ratio within {th.get('ratio')}, AOD <= {th.get('max_aod')},
  sun elevation >= {th.get('min_sun')} deg, NDVI correlation after alignment >= {th.get('min_ndvi_r')}, no unresolved shift,
  Landsat clear fraction (where Landsat has data) >= {th.get('min_landsat_clear')}, large shifts confirmed,
  haze: NDVI gain (mean EMIT NDVI / mean Landsat NDVI over Landsat NDVI > {hz.get('veg_landsat_ndvi')}) >= {hz.get('min_ndvi_gain')}.
  FLAGS only: blue_ok / red_ok / swir1_ok (ratio outside 0.85-1.15): blue {nb}, red {nr}, SWIR1 {ns} of {len(passp)} pairs;
  haze_mild (NDVI gain {hz.get('min_ndvi_gain')}-{hz.get('mild_below')}): {nmild} pairs. Drop or down-weight these if you need the cleanest set.

Known caveats
  - Several scenes share a cell: split training and validation BY cell_id, not by pair.
  - The EMIT pairs are humid tropical cells by design (profile humid_coastal); the Hyperion pairs cover the other climates.
  - EMIT reads brighter than Landsat in blue (median ratio ~1.36) and red (~1.1); which sensor is closer to the truth is not established.
  - EMIT position offsets of up to 6 px (360 m) were found and corrected (large_shift = True: {big if big else 'none'}); cause not established.
  - EMIT pairs are 60 m; the Hyperion pairs are 30 m.
  - 'humid_coastal' is a climate/land-cover profile; the cells are not necessarily on the coast.
  - PASS means "passed these checks", not perfect data.
"""
open(f"{OUT}/README.txt", "w").write(readme)

# ---- 5) summary ----
print("\nPASS pairs by source:", passp["source"].value_counts(dropna=False).to_dict(), "| by climate group:", passp["climate_group"].value_counts(dropna=False).to_dict())
print(f"cells: {n_cells} | pairs per cell: {passp['cell_id'].value_counts().value_counts().sort_index().to_dict()}")
print(f"flags among delivered pairs: blue {nb}, red {nr}, swir1 {ns}, haze_mild {nmild} | large shifts: {big}")
size = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(OUT) if "_to_delete" not in dp for f in fs) / 1e9
print(f"PASSED-PAIRS size: {size:.2f} GB | location: {OUT}")
if problems: print("\nPROBLEMS:", *problems, sep="\n  ")

# Design choices and the evidence for them
Each choice was made from what the data showed. Where a published source defines a fact, it is cited. Where no source exists
(thresholds, grid, caps), the choice is stated as a **design decision** with its evidence. The code and output of every test are in the archive
notebook `EMIT_LANDSAT.ipynb` (cell names in brackets). Every pair stores its QC numbers in `pair_info.json`, so thresholds can be changed
and the pairs re-labelled by re-running B1 + B2.

| Choice | Reason / evidence from this project | Source |
|---|---|---|
| EMIT L2A surface reflectance as delivered (no extra atmospheric correction) | L2A is already surface reflectance; reported error about ±1% against field spectra | EMIT L2A User Guide; ATBD; Coleman et al. 2024 |
| Earth Engine `NASA/EMIT/L2A/RFL` instead of NASA NetCDF downloads | Orthorectified, with reflectance, flags, AOD, water vapour; 36 of 40 handover scenes found by timestamp (E-01) | Earth Engine catalog |
| Drop bands 127-141 and 187-212 (+3 band margin): 232 of 285 kept | Both gaps have reflectance <= 0.01 or missing in the mean spectra of 36 scenes (E-03); water-vapour absorption | Design decision; standard physics |
| Valid pixel = `cloud_flag == 0` and >= 80% of kept bands > 0 | Dilated-cloud / aggregate flags marked 97% of a clear scene (24 Nov 2025; E-03). The dilated flag draws circles about 3 km wide around small clouds; with it, 11 clear PASS pairs would have failed (E-12b) | Design decision (visual + Landsat check) |
| EMIT native 60 m grid, Landsat averaged onto it | 60 m is EMIT's real resolution; interpolating EMIT to 30 m would invent detail | EMITL2ARFL product page |
| More scenes: humid_coastal profile, <= 4 scenes per cell, sun >= 25 deg, AOD <= 0.8, valid >= 0.5, cloud flag <= 0.3 (A3) | Kerala-like cells first; the cap limits near-duplicate dates per cell. 365 of 378 unique scenes passed the pre-filter | Design decision |
| Shift search +-8 px; shift only if best r > 0.6 and gain > 0.03 | Pairs 27-29 (Dec 2025, Kerala) matched Landsat only at 4-5 px (NDVI r 0.13-0.21 -> 0.81-0.84; E-07) | Design decision (data-driven) |
| Shift > 3 px needs >= 2 of 3 independent signals (red, SWIR1, brightness) within 1 px | Turned the visual check into a rule (E-08b: pairs 27-29 confirmed 3/3, controls at (0,0)). Full run: all 6 large shifts confirmed 3/3, in Kerala, Karnataka and the Philippines | Design decision |
| Residual cloud / shadow mask relative to the pair's own median ratio (5x5 median diff > 0.06, factor 1.5, 2 px buffer) | EMIT's cloud flag misses cloud edges and shadows (pair 58: white cloud counted as valid). Red/cyan masks match visible clouds and shadows offset in the same direction; NDVI r rises (pair 58: 0.78 -> 0.91, pair 41: 0.64 -> 0.82; E-12c). A fixed blue-ratio rule (as for Hyperion) would hit clear pixels because EMIT blue is ~1.36x Landsat. Buffer 1 -> 2 px after thin cloud edges slipped through (pair 248) | Design decision (data-driven); same idea as the Hyperion v2.2 haze mask |
| Scene haze rule: NDVI gain >= 0.80; 0.80-0.85 flagged `haze_mild` | Whole-scene haze is invisible to the pixel mask (pair 74: milky image, gain 0.51, but NDVI r 0.87 and NIR ratio 0.99). Gain correlates with the blue ratio (r = -0.83) but not with EMIT AOD (0.21), so the AOD rule cannot catch it (E-12d). 0.75 was tried first; pairs at 0.76 (254, 250, 323) still looked hazy (E-13 figure) | Design decision (visual) |
| PASS rules: joint coverage >= 0.5, NIR ratio 0.85-1.15, NDVI r >= 0.7, AOD <= 0.8, sun >= 25 deg, Landsat clear >= 0.85 | Same rules as the Hyperion pairs (set from the pilot pairs); not from the literature | Design decision |
| Blue / red / SWIR1 ratios are flags, not failures | EMIT blue is ~1.36x Landsat in almost every pair; failing on it would remove nearly all pairs | Design decision |

## QC history
| version | cell (archive) | change | result |
|---|---|---|---|
| v1 | E-05 | pilot rules, shift search +-3 px | 35 handover pairs |
| v2 | E-08 | joint coverage, all six ratios, search +-8 px | 33 PASS of 35 |
| v2.1 | E-12 | 365 extra scenes; large-shift confirmation rule; Koppen climate group | first 84 exports: 68 PASS |
| v2.2 | E-13 | residual cloud / shadow mask (1 px buffer); haze rule NDVI gain >= 0.75 | 274 PASS of 400 in 168 cells |
| v2.3 | E-13 / B1 | haze rule 0.80; cloud / shadow buffer 2 px | see B1 output |

## Limits to state openly
- All EMIT pairs are humid tropical (Koppen A) by design; Hyperion covers the arid, temperate and cold groups.
- Several scenes share a cell: training and validation must be split **by cell**, not by pair.
- The NDVI-gain test is slightly stricter on sparse, dry vegetation, because EMIT's small red excess (~1.1x) lowers NDVI proportionally more there.
- 41 pairs fail because EMIT NIR is > 15% above Landsat NIR (v2.2); the cause (view / sun geometry or data) is not established.
- EMIT position offsets up to 6 px (360 m) were found and corrected; their cause is not established.
- PASS means "passed these checks", not "perfect data". All thresholds are design choices.
- Hyperion pairs have a per-pixel haze mask but no scene-level haze check yet (to do: same NDVI-gain test).

## References (checked)
- NASA LP DAAC. *EMIT L2A Estimated Surface Reflectance and Uncertainty and Masks 60 m V001* (EMITL2ARFL). DOI 10.5067/EMIT/EMITL2ARFL.001.
- *EMIT L2A User Guide*, v1.0, JPL D-107863 (2022).
- Thompson, D.R., et al. (2020). *EMIT L2A Algorithm: Surface Reflectance and Scene Content Masks, Theoretical Basis*. JPL D-104255, v1.0.
- Coleman, R.W., Thompson, D.R., et al. (2024). An accuracy assessment of the surface reflectance product from the EMIT imaging spectrometer. *Remote Sensing of Environment*.
- Google Earth Engine Data Catalog: `NASA/EMIT/L2A/RFL`.
- Beck, H.E., et al. Koppen-Geiger climate classification maps (climate groups).

<!-- ADD AFTER CHECKING THE EXACT CITATIONS: USGS Landsat Collection 2 Level-2 Science Product Guide (scale factors, QA_PIXEL); Fmask cloud buffer (Zhu & Woodcock) -->